# Separate real tomato dataset experiment

This notebook reads the run output from `artifacts/public_tomato/metrics.json`. It is a **storage-scenario** experiment (Tc / Trt / Trtd), not a crop-quality or spoilage model. The 52 random tomatoes are used for cross-validation; a separate nine-tomato continuous cohort is evaluated once at its first recorded day. No records are merged with the five-crop simulator or potato histories.

Re-run from the project folder with `.\.venv\Scripts\python.exe -m crop_prototype.cli public-tomato`.

In [1]:
import json
from pathlib import Path
import pandas as pd

report = json.loads(Path("artifacts/public_tomato/metrics.json").read_text(encoding="utf-8"))
cv = pd.DataFrame([{
    "Inputs": "12 MQ" if m["feature_set"] == "mq_12" else "MQ + weight/color",
    "Model": m["model"],
    "Validation": "Leave one day out" if m["validation_protocol"].startswith("leave-one") else "Stratified 5-fold",
    "Macro F1": m["macro_f1_oof"], "Accuracy": m["accuracy_oof"],
    "Log loss": m["log_loss_oof"], "Brier": m["multiclass_brier_oof"]
} for m in report["modes"]])
display(cv.round(3))

,Inputs,Model,Validation,Macro F1,Accuracy,Log loss,Brier
0,12 MQ,majority,Leave one day out,0.171,0.346,1.101,0.668
1,12 MQ,majority,Stratified 5-fold,0.227,0.327,1.101,0.669
2,12 MQ,random_forest,Leave one day out,0.514,0.519,1.024,0.594
3,12 MQ,random_forest,Stratified 5-fold,0.558,0.558,1.628,0.562
4,12 MQ,xgboost,Leave one day out,0.498,0.519,1.005,0.601
5,12 MQ,xgboost,Stratified 5-fold,0.547,0.558,0.987,0.579
6,12 MQ,svc,Leave one day out,0.315,0.346,1.115,0.678
7,12 MQ,svc,Stratified 5-fold,0.404,0.423,1.088,0.658
8,MQ + weight/color,majority,Leave one day out,0.171,0.346,1.101,0.668
9,MQ + weight/color,majority,Stratified 5-fold,0.227,0.327,1.101,0.669


In [2]:
external = pd.DataFrame([{
    "Inputs": "12 MQ" if m["feature_set"] == "mq_12" else "MQ + weight/color",
    "Model": m["model"], "Test tomatoes": m["n_independent_tomatoes"],
    "Macro F1": m["macro_f1"], "Accuracy": m["accuracy"],
    "Log loss": m["log_loss"], "Brier": m["multiclass_brier"]
} for m in report["independent_continuous_test"]])
display(external.round(3))

,Inputs,Model,Test tomatoes,Macro F1,Accuracy,Log loss,Brier
0,12 MQ,majority,9,0.167,0.333,1.100,0.668
1,12 MQ,random_forest,9,0.278,0.333,1.172,0.746
2,12 MQ,xgboost,9,0.278,0.333,1.247,0.770
3,12 MQ,svc,9,0.533,0.556,1.078,0.655
4,MQ + weight/color,majority,9,0.167,0.333,1.100,0.668
5,MQ + weight/color,random_forest,9,0.356,0.444,1.181,0.691
6,MQ + weight/color,xgboost,9,0.278,0.333,1.229,0.707
7,MQ + weight/color,svc,9,0.345,0.444,1.012,0.616


## Interpretation

Treat the nine-tomato result as a small, within-study external check, not independent crop validation. Its tests show no reliable advantage from combining MQ with weight/color, and the target is storage scenario. The real-data record provides no acceptable/warning/spoiled labels or time-to-first-failure intervals.